In [1]:
from sympy import Function, Symbol, symbols, simplify, Eq, Symbol, latex, pprint, collect, expand
from sympy import init_printing
from IPython.display import display, Math

init_printing(use_latex=True)

In [2]:
import re
from IPython.display import display, Math

def color_terms(latex_str):
    latex_str = re.sub(
        r'(\\phi_\{REF\}\{\\left\(.+?\\right\)\})',
        r'{\\color{purple} \1}',
        latex_str
    )
    # Color all q_i(t) terms orange
    latex_str = re.sub(
        r'(q_\{(\d+)\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{orange} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([A])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Cyan} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([B])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{Aquamarine} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([C])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{SpringGreen} \1{\\left(\3\\right)}}',
        latex_str
    )
    return latex_str

In [3]:
# ── time variable ─────────────────────────────────────────────────────────────
t = Symbol('t')

# ── delay parameters ──────────────────────────────────────────────────────────
tau12, tau21, tau13, tau31, tau23, tau32 = symbols(
    r'\tau_{12} \tau_{21} \tau_{13} \tau_{31} \tau_{23} \tau_{32}',
    real=True, positive=True
)

dtau12, dtau21, dtau13, dtau31, dtau23, dtau32 = symbols(
    r'\dot\tau_{12} \dot\tau_{21} \dot\tau_{13} \dot\tau_{31} \dot\tau_{23} \dot\tau_{32}',
    real=True
)

# ── laser angular frequencies (physical + measurement offsets) ────────────────
omega1, omega2, omega3 = symbols(r'\omega_1 \omega_2 \omega_3', real=True)
omega1m, omega2m, omega3m = symbols(r'\omega_1^m \omega_2^m \omega_3^m', real=True)

For now, we assume frequencies to be constant in time. Hartwig et. al. did a bit of analysis how this affects the clock noise removal.

In [4]:
# ── abstract time-dependent functions ─────────────────────────────────────────
phi1  = Function(r'\phi_1')   # laser phase noise, s/c 1
phi2  = Function(r'\phi_2')
phi3  = Function(r'\phi_3')
phiREF  = Function(r'\phi_{REF}')

epsilonA = Function(r'\epsilon_A')  # clock noise, s/c 1 (master)
epsilonB = Function(r'\epsilon_B')  # clock noise, s/c 2
epsilonC = Function(r'\epsilon_C')  # clock noise, s/c 3

omegaREFA, omegaREFB, omegaREFC = symbols(r'\omega^{REF}_A \omega^{REF}_B \omega^{REF}_C', real=True)

q1 = Function('q_1')
q2 = Function('q_2')
q3 = Function('q_3')

In [ ]:
tau = {
    (1,2): tau12, (2,1): tau21,
    (1,3): tau13, (3,1): tau31,
    (2,3): tau23, (3,2): tau32,
}

dtau = {
    (1,2): dtau12, (2,1): dtau21,
    (1,3): dtau13, (3,1): dtau31,
    (2,3): dtau23, (3,2): dtau32,
}


def D(expr, tau_key):
    """
    Apply a single delay for link tau_key.
    - If tau_key is a tuple (i,j): uses time-varying delay tau_ij(t) = tau_ij^0 + dtau_ij * t
    - If tau_key is a sympy symbol: applies it as a constant delay
    """
    if isinstance(tau_key, tuple):
        tau_val = tau[tau_key] + dtau[tau_key] * t
    else:
        tau_val = tau_key
    return expr.subs(t, t - tau_val)


def Dn(expr, *tau_keys):
    """
    Apply a sequence of delays in order, each evaluated at the
    already-shifted time from previous delays.
    
    Usage:
        Dn(expr, (1,2), (2,3))        # D_12 then D_23, time-varying
        Dn(expr, (1,2), tau23)        # D_12 time-varying, tau23 constant
        Dn(expr, tau12, tau23)        # both constant
    
    The delays are applied left-to-right, so Dn(f, (1,2), (2,3)) gives
    f(t - tau_12(t) - tau_23(t - tau_12(t))), which for linearly
    changing delays expands as:
        f(t - tau_12^0 - dtau_12*t - tau_23^0 - dtau_23*(t - tau_12^0 - dtau_12*t))
    """
    for tau_key in tau_keys:
        expr = D(expr, tau_key)
    return expr

In [7]:
# Map spacecraft index → (phi, q, omega, omega_m, clock_epsilon)
sc = {
    1: (phi1, q1, omega1, omega1m, epsilonA, omegaREFA),
    2: (phi2, q2, omega2, omega2m, epsilonB, omegaREFB),
    3: (phi3, q3, omega3, omega3m, epsilonC, omegaREFC),
}


eta    = {}
etaSB  = {}
etaU  = {} # unmixed
REF  = {}
r = {}

include_phi = True 
include_clock_noise = True
include_board_jitter = True  

for (i, j) in tau:
    phi_i, q_i, om_i, omm_i, eps_i, omegaREFi = sc[i]
    phi_j, q_j, om_j, omm_j, eps_j, omegaREFj = sc[j]

    phi_terms = (D(phi_j(t) - phiREF(t), (i,j)) - (phi_i(t) - phiREF(t))) if include_phi else 0

    clock_terms_C = (-(om_j - om_i) * q_i(t)) if include_clock_noise else 0
    clock_terms_SB = (-(om_j - om_i + omm_j - omm_i)*q_i(t)
                      - omm_i*q_i(t)
                      + omm_j * D(q_j(t), (i,j))) if include_clock_noise else 0

    board_terms_c  = (om_j * (eps_i(t) - D(eps_i(t), (i,j)))) if include_board_jitter else 0
    board_terms_SB = ((om_j + omm_j) * (eps_i(t) - D(eps_i(t), (i,j)))) if include_board_jitter else 0

    eta[(i,j)] = (phi_terms + clock_terms_C + board_terms_c)

    etaSB[(i,j)] = collect(expand(phi_terms + clock_terms_SB + board_terms_SB), [q_i(t), q_j(t)])

    r[(i,j)] = simplify((eta[(i,j)] - etaSB[(i,j)]) / omm_j)

    #etaU[(i,j)] = (D(phi_j(t), t_ij) - om_j  * q_i(t) + (om_j * (eps_i(t) - D(eps_i(t), t_ij))))
    
    REF[(i,j)] = ( - int(include_clock_noise)*q_j(t) + int(include_board_jitter) * eps_i(t) ) # REF measured by the other PM, missing omegaREFi  *

The $\epsilon(t)$ terms originate from clock noise on the delay line boards which we will use three of. So we have A, B and C. Each corresponds to one spacecraft so the board A is where we apply delays 2 $\rightarrow$ 1 and 3 $\rightarrow$ 1. The $q(t)$ terms come from the Mokus. These are LISA-like, as in LISA will see similat clock noise coupling from the PM. All clock terms are given w.r.t the global clock, which we won't really ever know.

In [8]:
if True:
    for (i, j) in tau:
        display(Math(r'\eta_{' + str(i) + str(j) + '} = ' + color_terms(latex(eta[(i,j)]))))
        #display(Math(r'\eta_{' + str(i) + str(j) + '}^{U} = ' + color_terms(latex(etaU[(i,j)]))))
        display(Math(r'\eta_{' + str(i) + str(j) + '}^{SB} = ' + color_terms(latex(etaSB[(i,j)]))))
        display(Math(r'\frac{r_{' + str(i) + str(j) + r'}}{\omega_{' + str(j) + '}^m} = ' + color_terms(latex(r[(i,j)]))))
        display(Math(r'\frac{REF_{' + str(i) + str(j) + r'}}{ \omega_{' + str(j) + '}^m} = ' + color_terms(latex(REF[(i,j)]))))
        print("------------------------------------------------------")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


In [ ]:
# TDI gen 2, Michelson X
# ── TDI Generation 2 Michelson X, centered on SC1 ────────────────────────────
# P12 = -(1 - D131 - D13121 + D1213131)
# P13 =  (1 - D121 - D12131 + D1312121)
# P21 = P12 * D12
# P31 = P13 * D13
# P23 = P32 = 0

def P12_X2(expr):
    return -(  expr
             - Dn(expr, (1,3),(3,1))
             - Dn(expr, (1,3),(3,1),(2,1))
             + Dn(expr, (1,2),(2,1),(3,1),(2,1),(3,1)))

def P13_X2(expr):
    return  (  expr
             - Dn(expr, (1,2),(2,1))
             - Dn(expr, (1,2),(2,1),(3,1))
             + Dn(expr, (1,3),(3,1),(2,1),(2,1),(3,1)))

def P21_X2(expr): return P12_X2(Dn(expr, (1,2)))
def P31_X2(expr): return P13_X2(Dn(expr, (1,3)))
def P23_X2(expr): return 0
def P32_X2(expr): return 0

# X2    
X2 = collect(expand(P13(eta[(1,3)] + D(eta[(3,1)], tau13))+ P12(eta[(1,2)] + D(eta[(2,1)], tau12))), [q1(t), q2(t), q3(t), epsilonA(t), epsilonB(t), epsilonC(t)])

In [ ]:
display(Math(r'X_2 = ' + color_terms(latex(X2)))) 